# Does exercise Elo depend on work mode?

This notebook tests **measurement invariance** of the current exercise Elo. It recalibrates the same model on three datasets:

1. all first attempts (the existing `agg_exercise_elo.parquet` artifact);
2. playlist first attempts only;
3. ZPDES first attempts only.

It then compares only exercise contexts calibrated in both playlist and ZPDES. In the current model, an exercise context is the tuple `(module_code, objective_id, activity_id, exercise_id)`, not the raw exercise id alone.

A persistent disagreement on well-observed shared contexts is evidence that relative exercise difficulty is not invariant across work modes. It is not, by itself, proof that the all-mode Elo is unusable: selection of students, sparse observations, different learning/scaffolding conditions, and the arbitrary anchoring of separately fitted Elo scales can also create differences.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = next(
        parent for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]
        if (parent / 'pyproject.toml').exists()
    )
for import_root in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from visu2.config import get_settings
from visu2.derive_common import ELO_SCALE
from scripts.compare_work_mode_exercise_elo import (
    align_elo_columns_within_module,
    build_calibration_coverage,
    build_matched_work_mode_elo,
    build_module_agreement_summary,
    build_work_mode_agreement_metrics,
    calibrate_exercise_elo_by_work_mode,
    summarize_elo_pair,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

In [2]:
SOURCE_ID = 'mia'
LEFT_MODE = 'playlist'
RIGHT_MODE = 'zpdes'
MIN_ATTEMPT_THRESHOLDS = (1, 10, 25, 50, 100)
PRIMARY_MIN_ATTEMPTS = 25
RECALIBRATE_MODE_ELO = False  # Existing cached fits are reused when available.
RUN_SHARED_STUDENT_SENSITIVITY = True  # Cached when available; set False to skip.

settings = get_settings(SOURCE_ID)
FACT_PATH = settings.artifacts_derived_dir / 'fact_attempt_core.parquet'
ALL_ELO_PATH = settings.artifacts_derived_dir / 'agg_exercise_elo.parquet'
OUTPUT_DIR = settings.artifacts_reports_dir / 'work_mode_exercise_elo_invariance'
MODE_ELO_PATHS = {
    mode: OUTPUT_DIR / f'agg_exercise_elo_{mode}.parquet'
    for mode in (LEFT_MODE, RIGHT_MODE)
}
SHARED_STUDENT_ELO_PATHS = {
    mode: OUTPUT_DIR / f'agg_exercise_elo_{mode}_shared_students.parquet'
    for mode in (LEFT_MODE, RIGHT_MODE)
}

assert FACT_PATH.exists(), f'Missing attempt artifact: {FACT_PATH}'
assert ALL_ELO_PATH.exists(), f'Missing all-mode Elo artifact: {ALL_ELO_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    'fact_path': str(FACT_PATH),
    'all_mode_elo_path': str(ALL_ELO_PATH),
    'mode_elo_cache_dir': str(OUTPUT_DIR),
}

{'fact_path': 'C:\\Users\\ocler\\Documents\\Académique\\Inria\\GAIMHE\\Code\\visu2\\artifacts\\sources\\mia\\artifacts\\derived\\fact_attempt_core.parquet',
 'all_mode_elo_path': 'C:\\Users\\ocler\\Documents\\Académique\\Inria\\GAIMHE\\Code\\visu2\\artifacts\\sources\\mia\\artifacts\\derived\\agg_exercise_elo.parquet',
 'mode_elo_cache_dir': 'C:\\Users\\ocler\\Documents\\Académique\\Inria\\GAIMHE\\Code\\visu2\\artifacts\\sources\\mia\\artifacts\\reports\\work_mode_exercise_elo_invariance'}

## Exact calibration reused here

The mode-specific fits call the same production function as the all-mode artifact. For each module, it keeps valid first attempts, jointly estimates student ability and context difficulty with regularization, and re-centers mean exercise Elo to 1500. No success-rate shortcut or new Elo definition is introduced.

In [3]:
fact = pl.scan_parquet(FACT_PATH)
all_mode_elo = pl.read_parquet(ALL_ELO_PATH)

source_summary = {
    'attempt_rows': fact.select(pl.len()).collect().item(),
    'all_mode_calibrated_contexts': all_mode_elo.filter(pl.col('calibrated')).height,
}
source_summary

{'attempt_rows': 6404817, 'all_mode_calibrated_contexts': 18556}

In [4]:
mode_expr = pl.col('work_mode').cast(pl.Utf8).str.strip_chars().str.to_lowercase()
valid_first_attempts = fact.filter(
    (pl.col('attempt_number') == 1)
    & pl.col('data_correct').is_not_null()
    & pl.col('user_id').is_not_null()
    & mode_expr.is_in([LEFT_MODE, RIGHT_MODE])
).with_columns(mode_expr.alias('work_mode_normalized'))

attempt_audit = (
    valid_first_attempts
    .group_by('work_mode_normalized')
    .agg(
        pl.len().alias('first_attempts'),
        pl.col('user_id').n_unique().alias('students'),
        pl.struct(['module_code', 'objective_id', 'activity_id', 'exercise_id'])
        .n_unique()
        .alias('observed_contexts'),
    )
    .sort('work_mode_normalized')
    .collect()
    .to_pandas()
)
student_overlap = (
    valid_first_attempts
    .select(['user_id', 'work_mode_normalized'])
    .unique()
    .group_by('user_id')
    .agg(pl.col('work_mode_normalized').n_unique().alias('modes_observed'))
    .group_by('modes_observed')
    .agg(pl.len().alias('students'))
    .sort('modes_observed')
    .collect()
    .to_pandas()
)
display(Markdown('### Calibration population by mode'))
display(attempt_audit)
display(Markdown('### Student overlap (2 means observed in both modes)'))
display(student_overlap)

### Calibration population by mode

,work_mode_normalized,first_attempts,students,observed_contexts
0,playlist,1427818,15853,16032
1,zpdes,3196033,28485,18393


### Student overlap (2 means observed in both modes)

,modes_observed,students
0,1,31574
1,2,6382


In [5]:
cache_is_complete = all(path.exists() for path in MODE_ELO_PATHS.values())
if RECALIBRATE_MODE_ELO or not cache_is_complete:
    print('Calibrating playlist-only and ZPDES-only exercise Elo...')
    mode_elos = calibrate_exercise_elo_by_work_mode(
        fact,
        settings,
        work_modes=(LEFT_MODE, RIGHT_MODE),
    )
    for mode, frame in mode_elos.items():
        frame.write_parquet(MODE_ELO_PATHS[mode])
        print(f'Cached {mode}: {MODE_ELO_PATHS[mode]}')
else:
    print('Loading cached mode-specific Elo fits...')
    mode_elos = {mode: pl.read_parquet(path) for mode, path in MODE_ELO_PATHS.items()}

coverage = build_calibration_coverage(all_mode_elo, mode_elos)
coverage

Loading cached mode-specific Elo fits...


,calibration,calibrated_contexts,modules,first_attempts,median_attempts_per_context,p10_attempts_per_context,p90_attempts_per_context
0,all,18556,27,5345539,115.000,22.000,758.000
1,playlist,15821,25,1424601,36.000,2.000,257.000
2,zpdes,18393,27,3196033,59.000,8.000,454.000


In [6]:
matched = build_matched_work_mode_elo(
    all_mode_elo,
    mode_elos,
    left_mode=LEFT_MODE,
    right_mode=RIGHT_MODE,
)
metrics = build_work_mode_agreement_metrics(
    matched,
    thresholds=MIN_ATTEMPT_THRESHOLDS,
    left_mode=LEFT_MODE,
    right_mode=RIGHT_MODE,
)

matched.to_parquet(OUTPUT_DIR / 'matched_playlist_zpdes_exercise_elo.parquet', index=False)
metrics.to_csv(OUTPUT_DIR / 'agreement_metrics.csv', index=False)

overlap_summary = pd.DataFrame(
    {
        'shared_calibrated_contexts': [len(matched)],
        f'shared_contexts_with_at_least_{PRIMARY_MIN_ATTEMPTS}_attempts_per_mode': [
            int((matched['min_mode_attempts'] >= PRIMARY_MIN_ATTEMPTS).sum())
        ],
        'modules_with_shared_contexts': [matched['module_code'].nunique()],
    }
)
overlap_summary

,shared_calibrated_contexts,shared_contexts_with_at_least_25_attempts_per_mode,modules_with_shared_contexts
0,15697,7978,24


## Why two comparison scales are reported

A separately fitted module Elo scale has an arbitrary additive origin. Both fits are set to mean 1500, but playlist and ZPDES can contain different non-shared contexts. Their shared exercises can therefore have different raw means even when their relative difficulty is identical.

- **Raw** retains that shift, but it mixes possible mode divergence with the anchoring/item-mix artifact.
- **Shared-module aligned** subtracts each mode's mean over the retained shared contexts in each module and adds 1500. Alignment is recomputed after each support threshold is applied, so excluded sparse contexts cannot shift the retained comparison. This tests whether the *relative ordering and spacing* of the same contexts agree. It is the primary diagnostic.

The average signed difference is zero within each module by construction on every aligned analysis subset. Therefore, use mean absolute difference (MAE), RMSE, correlations, and concordance—not the signed mean—as the evidence of divergence.

In [7]:
metric_columns = [
    'scale',
    'min_attempts_per_mode',
    'contexts',
    'modules',
    'mean_absolute_difference',
    'median_absolute_difference',
    'rmse',
    'p90_absolute_difference',
    'within_50_elo_percent',
    'within_100_elo_percent',
    'pearson_correlation',
    'spearman_correlation',
    'lin_concordance',
]
display(metrics[metric_columns])

primary = metrics.loc[
    (metrics['scale'] == 'shared_module_aligned')
    & (metrics['min_attempts_per_mode'] == PRIMARY_MIN_ATTEMPTS)
].iloc[0]
display(
    Markdown(
        f"**Primary comparison ({int(primary['contexts']):,} shared contexts, "
        f"at least {PRIMARY_MIN_ATTEMPTS} first attempts in each mode):** "
        f"median absolute gap **{primary['median_absolute_difference']:.1f} Elo**, "
        f"MAE **{primary['mean_absolute_difference']:.1f}**, "
        f"RMSE **{primary['rmse']:.1f}**, "
        f"Spearman **{primary['spearman_correlation']:.3f}**, "
        f"Lin concordance **{primary['lin_concordance']:.3f}**."
    )
)

,scale,min_attempts_per_mode,contexts,modules,mean_absolute_difference,median_absolute_difference,rmse,p90_absolute_difference,within_50_elo_percent,within_100_elo_percent,pearson_correlation,spearman_correlation,lin_concordance
0,raw,1,15697,24,115.349,90.847,150.027,247.280,28.910,53.978,0.718,0.681,0.718
1,raw,10,10367,24,102.253,80.954,133.692,219.892,32.459,59.197,0.807,0.784,0.805
2,raw,25,7978,24,95.044,74.803,124.747,203.746,34.796,62.710,0.841,0.826,0.838
3,raw,50,5356,22,88.809,69.421,116.908,191.702,36.837,66.113,0.874,0.868,0.870
4,raw,100,3427,20,83.944,65.377,111.371,180.103,39.393,68.865,0.897,0.897,0.892
5,shared_module_aligned,1,15697,24,115.163,90.934,149.850,246.541,28.986,53.934,0.719,0.682,0.719
6,shared_module_aligned,10,10367,24,101.439,80.354,132.856,216.081,32.594,59.651,0.807,0.786,0.805
7,shared_module_aligned,25,7978,24,93.489,73.443,122.761,199.605,35.021,63.261,0.843,0.830,0.841
8,shared_module_aligned,50,5356,22,86.550,68.595,113.875,185.100,37.509,66.990,0.876,0.873,0.873
9,shared_module_aligned,100,3427,20,81.918,64.425,108.974,175.644,40.706,69.886,0.895,0.897,0.891


**Primary comparison (7,978 shared contexts, at least 25 first attempts in each mode):** median absolute gap **73.4 Elo**, MAE **93.5**, RMSE **122.8**, Spearman **0.830**, Lin concordance **0.841**.

In [8]:
elo_gap_meaning = pd.DataFrame({'absolute_elo_gap': [25, 50, 100, 200]})
elo_gap_meaning['success_odds_ratio_at_equal_student_ability'] = (
    10 ** (elo_gap_meaning['absolute_elo_gap'] / ELO_SCALE)
)
elo_gap_meaning

,absolute_elo_gap,success_odds_ratio_at_equal_student_ability
0,25,1.155
1,50,1.334
2,100,1.778
3,200,3.162


The table above translates an Elo gap into its model scale. For example, a 100-point difference corresponds to about 1.78 times different predicted success odds for the same student ability. This gives MAE and RMSE a concrete interpretation; there is no universal pass/fail threshold.

In [9]:
plot_data = matched.loc[matched['min_mode_attempts'] >= PRIMARY_MIN_ATTEMPTS].copy()
plot_data = align_elo_columns_within_module(
    plot_data,
    ['exercise_elo_playlist', 'exercise_elo_zpdes', 'exercise_elo_all'],
)
plot_data['elo_difference_aligned'] = (
    plot_data['exercise_elo_zpdes_aligned']
    - plot_data['exercise_elo_playlist_aligned']
)
plot_data['absolute_difference_aligned'] = plot_data['elo_difference_aligned'].abs()
plot_data['elo_mean_aligned'] = 0.5 * (
    plot_data['exercise_elo_zpdes_aligned']
    + plot_data['exercise_elo_playlist_aligned']
)
plot_data['support_for_plot'] = np.sqrt(plot_data['min_mode_attempts'])

scatter = px.scatter(
    plot_data,
    x='exercise_elo_playlist_aligned',
    y='exercise_elo_zpdes_aligned',
    color='module_code',
    size='support_for_plot',
    size_max=18,
    opacity=0.55,
    hover_data=[
        'exercise_id', 'objective_id', 'activity_id',
        'calibration_attempts_playlist', 'calibration_attempts_zpdes',
        'elo_difference_aligned',
    ],
    labels={
        'exercise_elo_playlist_aligned': 'Playlist Elo (shared-context aligned)',
        'exercise_elo_zpdes_aligned': 'ZPDES Elo (shared-context aligned)',
    },
    title=f'Shared exercise-context Elo agreement (at least {PRIMARY_MIN_ATTEMPTS} attempts per mode)',
)
if not plot_data.empty:
    plot_min = float(
        plot_data[['exercise_elo_playlist_aligned', 'exercise_elo_zpdes_aligned']].min().min()
    )
    plot_max = float(
        plot_data[['exercise_elo_playlist_aligned', 'exercise_elo_zpdes_aligned']].max().max()
    )
    scatter.add_trace(
        go.Scatter(
            x=[plot_min, plot_max], y=[plot_min, plot_max],
            mode='lines', name='perfect agreement',
            line={'color': 'black', 'dash': 'dash'},
        )
    )
scatter.update_layout(legend_title_text='Module')
scatter.show()

In [10]:
mean_difference = plot_data['elo_difference_aligned'].mean()
sd_difference = plot_data['elo_difference_aligned'].std(ddof=1)
bland_altman = px.scatter(
    plot_data,
    x='elo_mean_aligned',
    y='elo_difference_aligned',
    color='module_code',
    opacity=0.55,
    hover_data=['exercise_id', 'min_mode_attempts'],
    labels={
        'elo_mean_aligned': 'Mean aligned Elo across modes',
        'elo_difference_aligned': 'ZPDES Elo - playlist Elo',
    },
    title='Bland-Altman view: does disagreement change with exercise difficulty?',
)
for value, label, dash in [
    (mean_difference, 'mean difference', 'solid'),
    (mean_difference + 1.96 * sd_difference, '+1.96 SD', 'dash'),
    (mean_difference - 1.96 * sd_difference, '-1.96 SD', 'dash'),
]:
    bland_altman.add_hline(y=value, line_dash=dash, annotation_text=label)
bland_altman.show()

In [11]:
aligned_metrics = metrics.loc[metrics['scale'] == 'shared_module_aligned'].copy()
threshold_figure = go.Figure()
threshold_figure.add_trace(
    go.Scatter(
        x=aligned_metrics['min_attempts_per_mode'],
        y=aligned_metrics['median_absolute_difference'],
        mode='lines+markers',
        name='Median absolute Elo gap',
    )
)
threshold_figure.add_trace(
    go.Scatter(
        x=aligned_metrics['min_attempts_per_mode'],
        y=aligned_metrics['mean_absolute_difference'],
        mode='lines+markers',
        name='Mean absolute Elo gap',
    )
)
threshold_figure.update_layout(
    title='Does disagreement persist when low-support contexts are removed?',
    xaxis_title='Minimum first attempts in each mode',
    yaxis_title='Absolute aligned Elo gap',
)
threshold_figure.show()
display(aligned_metrics[[
    'min_attempts_per_mode', 'contexts', 'median_absolute_difference',
    'mean_absolute_difference', 'spearman_correlation', 'lin_concordance',
]])

,min_attempts_per_mode,contexts,median_absolute_difference,mean_absolute_difference,spearman_correlation,lin_concordance
5,1,15697,90.934,115.163,0.682,0.719
6,10,10367,80.354,101.439,0.786,0.805
7,25,7978,73.443,93.489,0.830,0.841
8,50,5356,68.595,86.550,0.873,0.873
9,100,3427,64.425,81.918,0.897,0.891


In [12]:
module_summary = build_module_agreement_summary(
    matched, min_attempts_per_mode=PRIMARY_MIN_ATTEMPTS
)
module_summary.to_csv(OUTPUT_DIR / 'module_agreement_summary.csv', index=False)
display(Markdown('### Agreement by module'))
display(module_summary)

top_columns = [
    'module_code', 'module_label', 'objective_id', 'objective_label',
    'activity_id', 'activity_label', 'exercise_id', 'exercise_label',
    'exercise_elo_playlist_aligned', 'exercise_elo_zpdes_aligned',
    'elo_difference_aligned', 'absolute_difference_aligned',
    'calibration_attempts_playlist', 'calibration_attempts_zpdes',
]
top_divergences = (
    plot_data.sort_values('absolute_difference_aligned', ascending=False)
    .loc[:, [column for column in top_columns if column in plot_data.columns]]
    .head(30)
)
top_divergences.to_csv(OUTPUT_DIR / 'top_divergent_contexts.csv', index=False)
display(Markdown('### Most divergent well-observed contexts'))
display(top_divergences)

### Agreement by module

,module_code,module_label,contexts,mean_difference,mean_absolute_difference,rmse,pearson_correlation,spearman_correlation,lin_concordance
5,M104,Calcul littéral,126,0.000,158.460,188.938,0.545,0.521,0.485
21,M7,Syntaxe niveau 2,490,-0.000,124.747,164.453,0.637,0.581,0.636
9,M108,Algorithmique et programmation,132,-0.000,119.123,145.102,0.514,0.523,0.503
16,M2,Fluence de décodage de la lecture,147,-0.000,118.027,149.739,0.738,0.696,0.705
4,M103,Nombres et calculs,438,-0.000,113.970,142.520,0.810,0.795,0.802
0,M1,Réapprentissage des correspondances graphèmes-...,208,0.000,110.857,138.625,0.381,0.373,0.372
7,M106,Grandeurs et mesures,96,0.000,110.420,142.255,0.791,0.789,0.791
13,M14,Cohérence du texte,207,0.000,102.471,128.382,0.839,0.807,0.836
22,M8,Orthographe niveau 1,493,-0.000,100.798,132.424,0.900,0.853,0.896
10,M11,Lexique niveau 2,96,-0.000,98.562,124.521,0.700,0.654,0.698


### Most divergent well-observed contexts

,module_code,module_label,objective_id,objective_label,activity_id,activity_label,exercise_id,exercise_label,exercise_elo_playlist_aligned,exercise_elo_zpdes_aligned,elo_difference_aligned,absolute_difference_aligned,calibration_attempts_playlist,calibration_attempts_zpdes
9204,M13,Verbe niveau 2,a995b848-d365-4f71-a9a2-9e9bab6de86e,Concordance : subordonnée à l'indicatif,50d1366c-154c-4ff5-9847-38fdf93dce10,Concordance : princ. et subord. à l'indicatif ...,d3ecc6d0-f298-4bbe-9166-b9bb3dd83bed,d3ecc6d0-f298-4bbe-9166-b9bb3dd83bed,992.488,"1,654.659",662.171,662.171,87,29
14504,M7,Syntaxe niveau 2,fd9fe14b-3021-412f-8be4-dc2f2d78f4b0,Liens principale - circonstancielle,bc540a08-5dbf-413c-8d21-14a47284341c,Circonstancielle : place,901b6760-6ebd-41ea-a8ea-222cae588c68,901b6760-6ebd-41ea-a8ea-222cae588c68,"1,234.781","1,896.827",662.046,662.046,50,30
5113,M104,Calcul littéral,91f909ab-7d13-4eab-b59e-08fbb44ee674,S’appuyer sur des calculs d’aires pour compren...,3023716b-fc7b-4bd4-a5ae-0efca527b306,"Distributivité simple, somme en un produit (fa...",b14b20b6-5485-4cf8-84db-5157c56a7dc7,b14b20b6-5485-4cf8-84db-5157c56a7dc7,"2,004.609","1,399.693",-604.916,604.916,66,27
14734,M8,Orthographe niveau 1,6d4000fe-a2a0-4b82-b3f7-a21fae58a217,"Ambiguïtés graphiques (ou/où, er/é…)",f6e4812d-ba33-4020-bf16-d72cd65c9ae5,Identifier intuitivement la préposition « sans...,a4605f91-4139-4b77-a9ba-213ee06d02c7,a4605f91-4139-4b77-a9ba-213ee06d02c7,"1,304.916","1,896.796",591.880,591.880,30,148
1181,M1,Réapprentissage des correspondances graphèmes-...,f13f1fed-bbdc-49f5-90bd-ad7a03a56a25,Discrimination graphème/phonème simples et trè...,f8d1afdd-ff44-49c8-859b-4e0553ec5084,Distinction graphème/phonème : l / [l],894c4074-5802-4469-b319-b8d6f19054d0,894c4074-5802-4469-b319-b8d6f19054d0,"2,138.880","1,554.895",-583.985,583.985,66,480
11409,M2,Fluence de décodage de la lecture,2fd11e5d-810b-4662-9064-25eb469485b7,Améliorer la fluence de décodage​,998a239c-cfa2-4968-9300-74eb56acac32,Séquençage de lettres composant une syllabe (4...,f9081199-4d33-4c57-9284-3559ff3ade75,f9081199-4d33-4c57-9284-3559ff3ade75,"2,182.025","1,612.046",-569.980,569.980,47,41
14510,M7,Syntaxe niveau 2,fd9fe14b-3021-412f-8be4-dc2f2d78f4b0,Liens principale - circonstancielle,cc7ec02a-aa98-40be-b0df-23a196c6540f,Relation temporelle principale - subordonnée (...,082d1a98-2f0a-4bf2-9a87-c831c7ffba2d,082d1a98-2f0a-4bf2-9a87-c831c7ffba2d,"1,190.092","1,743.222",553.130,553.130,67,28
1692,M101,Réapprentissage du sens des nombres,0cc20a20-6eaa-423f-a53e-7f0032b3e985,Les puissances de 10 et la notation scientifique,fb91422d-ceda-4d0f-9906-29fd3e115018,Le centième d’un nombre entier (niveau 1),f1cdd4bf-4fab-44c8-ab73-c5e3638928c4,f1cdd4bf-4fab-44c8-ab73-c5e3638928c4,"2,089.103","1,536.015",-553.089,553.089,366,329
13354,M6,Syntaxe niveau 1,b94112e7-e953-48d6-99a0-c2686401caa5,Coordination : constituants / propositions,7f38faf6-fb72-4f75-9b07-ce4ef683f2d0,Coordination : constituants dissemblables,32ae21d3-43e7-4222-97e0-0b73e504424a,32ae21d3-43e7-4222-97e0-0b73e504424a,"1,264.155","1,798.183",534.028,534.028,34,327
1924,M101,Réapprentissage du sens des nombres,3a4414bb-6f2b-4855-a921-4fbfb467cd0e,Positionner des nombres entiers,3dc3e92e-7d98-4cb4-88ad-0f3c2f91f41f,Positionner sur un segment des nombres entiers...,a1d37f3c-59ad-48e2-afdd-82b5a28da42d,a1d37f3c-59ad-48e2-afdd-82b5a28da42d,"2,179.506","1,651.220",-528.285,528.285,418,1685


In [13]:
mode_to_all_tables = []
for mode in (LEFT_MODE, RIGHT_MODE):
    for scale, align_within_module in [
        ('raw', False), ('shared_module_aligned', True)
    ]:
        mode_to_all_tables.append(
            summarize_elo_pair(
                matched,
                left_column=f'exercise_elo_{mode}',
                right_column='exercise_elo_all',
                thresholds=MIN_ATTEMPT_THRESHOLDS,
                comparison=f'all_minus_{mode}',
                scale=scale,
                align_within_module=align_within_module,
            )
        )
mode_to_all = pd.concat(mode_to_all_tables, ignore_index=True)
display(
    mode_to_all.loc[
        (mode_to_all['scale'] == 'shared_module_aligned')
        & (mode_to_all['min_attempts_per_mode'] == PRIMARY_MIN_ATTEMPTS),
        [
            'comparison', 'contexts', 'mean_absolute_difference', 'rmse',
            'spearman_correlation', 'lin_concordance',
        ],
    ]
)

,comparison,contexts,mean_absolute_difference,rmse,spearman_correlation,lin_concordance
7,all_minus_playlist,7978,68.950,93.685,0.901,0.908
17,all_minus_zpdes,7978,38.581,54.271,0.963,0.967


## How to decide what the result means

Read the evidence in this order:

1. **Coverage:** there must be enough shared contexts and enough first attempts in both modes.
2. **Raw versus aligned:** a large raw gap that mostly disappears after shared-module alignment is mainly a scale-anchor/item-mix issue, not context-specific work-mode dependence.
3. **MAE and RMSE:** these measure the size of context-level disagreement without positive and negative gaps cancelling.
4. **Spearman:** asks whether exercise rankings agree.
5. **Lin concordance:** asks whether both ranking and numerical Elo values agree; it is stricter than correlation.
6. **Threshold stability:** if gaps shrink sharply as the minimum attempt count rises, sampling noise is a plausible explanation. If large gaps and weak concordance persist at 50 or 100 attempts per mode, the evidence for work-mode dependence is stronger.
7. **Module and Bland-Altman diagnostics:** check whether the issue is localized or changes systematically with difficulty.

A strong persistent aligned divergence means the pooled Elo should not be described as a completely work-mode-invariant exercise property. It may still be useful as a control variable, but conclusions should be checked with mode-specific Elo or, preferably, a joint item-response model containing work-mode-by-item differential item functioning effects.

## Optional robustness check: use only students observed in both modes

Different student populations can still confound two separate calibrations. The cell below repeats the mode-specific fits after restricting both datasets to students who have valid first attempts in both modes. This does not make exposure random, but it improves population comparability. The fitted artifacts are cached, and the check can be disabled in the parameters cell.

In [14]:
if RUN_SHARED_STUDENT_SENSITIVITY:
    shared_student_cache_complete = all(
        path.exists() for path in SHARED_STUDENT_ELO_PATHS.values()
    )
    if RECALIBRATE_MODE_ELO or not shared_student_cache_complete:
        both_mode_students = (
            valid_first_attempts
            .select(['user_id', 'work_mode_normalized'])
            .unique()
            .group_by('user_id')
            .agg(pl.col('work_mode_normalized').n_unique().alias('mode_count'))
            .filter(pl.col('mode_count') == 2)
            .select('user_id')
            .collect()
            .get_column('user_id')
            .to_list()
        )
        shared_student_fact = fact.filter(pl.col('user_id').is_in(both_mode_students))
        shared_student_elos = calibrate_exercise_elo_by_work_mode(
            shared_student_fact, settings, work_modes=(LEFT_MODE, RIGHT_MODE)
        )
        for mode, frame in shared_student_elos.items():
            frame.write_parquet(SHARED_STUDENT_ELO_PATHS[mode])
    else:
        shared_student_elos = {
            mode: pl.read_parquet(path)
            for mode, path in SHARED_STUDENT_ELO_PATHS.items()
        }
    shared_student_matched = build_matched_work_mode_elo(
        all_mode_elo, shared_student_elos, left_mode=LEFT_MODE, right_mode=RIGHT_MODE
    )
    shared_student_metrics = build_work_mode_agreement_metrics(
        shared_student_matched, thresholds=MIN_ATTEMPT_THRESHOLDS
    )
    shared_student_matched.to_parquet(
        OUTPUT_DIR / 'matched_playlist_zpdes_exercise_elo_shared_students.parquet',
        index=False,
    )
    shared_student_metrics.to_csv(
        OUTPUT_DIR / 'agreement_metrics_shared_students.csv', index=False
    )
    display(shared_student_metrics[metric_columns])
    shared_primary = shared_student_metrics.loc[
        (shared_student_metrics['scale'] == 'shared_module_aligned')
        & (
            shared_student_metrics['min_attempts_per_mode']
            == PRIMARY_MIN_ATTEMPTS
        )
    ].iloc[0]
    display(
        Markdown(
            f"**Shared-student comparison:** {int(shared_primary['contexts']):,} contexts; "
            f"median absolute gap **{shared_primary['median_absolute_difference']:.1f} Elo**, "
            f"MAE **{shared_primary['mean_absolute_difference']:.1f}**, "
            f"Spearman **{shared_primary['spearman_correlation']:.3f}**, "
            f"Lin concordance **{shared_primary['lin_concordance']:.3f}**."
        )
    )
else:
    display(Markdown('*Shared-student sensitivity skipped. Set `RUN_SHARED_STUDENT_SENSITIVITY = True` to run it.*'))

,scale,min_attempts_per_mode,contexts,modules,mean_absolute_difference,median_absolute_difference,rmse,p90_absolute_difference,within_50_elo_percent,within_100_elo_percent,pearson_correlation,spearman_correlation,lin_concordance
0,raw,1,11272,24,119.533,94.799,154.351,254.594,27.892,52.147,0.686,0.643,0.685
1,raw,10,6144,23,103.789,82.987,133.720,218.866,31.576,58.040,0.809,0.792,0.806
2,raw,25,3763,20,95.689,76.437,123.531,204.284,34.069,62.131,0.854,0.849,0.850
3,raw,50,2090,15,89.463,71.743,116.285,192.077,37.081,65.502,0.889,0.894,0.884
4,raw,100,678,10,80.703,64.559,105.141,169.943,40.855,70.501,0.909,0.902,0.905
5,shared_module_aligned,1,11272,24,119.184,94.982,153.797,253.527,27.883,52.076,0.687,0.645,0.686
6,shared_module_aligned,10,6144,23,102.485,82.007,132.174,216.527,32.292,58.480,0.811,0.795,0.808
7,shared_module_aligned,25,3763,20,94.697,75.732,122.007,199.808,34.627,62.556,0.853,0.849,0.850
8,shared_module_aligned,50,2090,15,88.414,70.040,115.257,189.440,36.699,65.885,0.888,0.893,0.883
9,shared_module_aligned,100,678,10,79.049,60.719,103.621,162.797,41.150,70.501,0.906,0.895,0.903


**Shared-student comparison:** 3,763 contexts; median absolute gap **75.7 Elo**, MAE **94.7**, Spearman **0.849**, Lin concordance **0.850**.

## Limitations

- The current Elo fit does not provide standard errors for individual exercise Elo estimates. Attempt thresholds reduce obvious instability but are not uncertainty intervals. A student-cluster bootstrap would be the natural next step.
- Separate calibrations cannot identify a single global ZPDES-versus-playlist difficulty shift: student and item locations can move together. The aligned analysis deliberately tests relative item invariance.
- Students and exercise exposure are not randomized across modes. The optional shared-student analysis reduces, but does not eliminate, selection effects.
- A mode-specific difference can reflect a real context effect—scaffolding, timing, learning within a sequence—not merely a defective Elo algorithm.

For a formal follow-up, fit one joint response model with common student ability, common item difficulty, and item-by-work-mode deviations. Those deviations directly test differential item functioning on one linked scale.